<a href="https://colab.research.google.com/github/RyanHadiA/Skripsi-TA/blob/main/v1_model_rule_based_Teks_pendek_(baseline_lexicon).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import re

#file_path = '/content/drive/MyDrive/Dataset Gabungan BIG5 biner/Teks panjang-pendek(192)/preprocessed_Gabungan Seluruh Dataset_long6046.csv'
file_path = '/content/drive/MyDrive/Dataset Gabungan BIG5 biner/Teks panjang-pendek(192)/preprocessed_Gabungan Seluruh Dataset_short31584.csv'
df = pd.read_csv(file_path)

# Hitung total jumlah data
total_samples = len(df)
print(f"Total data: {total_samples}\n")

# ==== Split Dataset =====
# Mengacak dataset terlebih dahulu
df = shuffle(df, random_state=42)

# Split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print jumlah tiap subset
print(f"Jumlah data pelatihan: {len(train_df)}")
print(f"Jumlah data validasi: {len(val_df)}")
print(f"Jumlah data pengujian: {len(test_df)}")

Total data: 31584

Jumlah data pelatihan: 25267
Jumlah data validasi: 3158
Jumlah data pengujian: 3159


# Definisi Model Rule-Based dengan Lexicon

Import library

In [ ]:
import json
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
import string
from typing import Optional

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


load lexicon dan tokenisasi NLTK

In [ ]:
# ======================================================
#         KONFIGURASI UNTUK MODEL RULE-BASED
# ======================================================
class ConfigRB:
    LEXICON_PATH = '/content/lexicon.json'
    MAX_LENGTH = 128

# ======================================================
#          LOAD LEXICON & DEFINISI FUNGSI
# ======================================================

# Load lexicon per trait menggunakan path dari ConfigRB
try:
    with open(ConfigRB.LEXICON_PATH, 'r') as f:
        lexicon = json.load(f)
    if lexicon:
        print("Lexicon loaded successfully. Berikut ini beberapa contoh kata dari leksikon:")
        for trait, words in lexicon.items():
            print(f"Trait '{trait}': {list(words.keys())[:10]}...")
except FileNotFoundError:
    print(f"Error: Lexicon file not found at {ConfigRB.LEXICON_PATH}. Please check the path in ConfigRB.")
    lexicon = None

if lexicon is not None:
    # Siapkan stop words sekali saja di luar fungsi
    stop_words = set(stopwords.words('english') + list(string.punctuation))

    # Definisi fungsi dengan parameter max_length
    def compute_trait_scores(text: str, max_length: Optional[int] = None) -> dict:
        """
        Menghitung skor trait, dengan opsi untuk membatasi jumlah token.
        """
        tokens = [t.lower() for t in word_tokenize(text)]
        tokens = [t for t in tokens if t not in stop_words]

        # Potong daftar token jika max_length diberikan
        if max_length is not None and max_length > 0:
            tokens = tokens[:max_length]

        scores = {trait: 0.0 for trait in lexicon.keys()}
        for token in tokens:
            for trait, mapping in lexicon.items():
                scores[trait] += mapping.get(token, 0.0)
        return scores
else:
    # Fungsi dummy jika lexicon gagal dimuat
    def compute_trait_scores(text: str, max_length: Optional[int] = None) -> dict:
        print("Lexicon not loaded due to missing file.")
        return {trait: 0.0 for trait in ['N', 'A', 'E', 'C', 'O']}

Lexicon loaded successfully. Berikut ini beberapa contoh kata dari leksikon:
Trait 'N': ['fucking', 'sick of', 'depression', 'fuck', 'depressed', 'xd', 'pissed', 'i hate', 'anymore', 'lonely']...
Trait 'A': ['fuck', 'fucking', 'shit', 'bitch', 'damn', 'hell', 'bitches', 'ass', 'fucked', 'wtf']...
Trait 'E': ['anime', 'xd', '>.<', 'o.o', 't_t', 'manga', '^', '> . >', '_', '^ ^']...
Trait 'C': ['xd', 'fucking', 'fuck', 'd:', 'fuck you', ':d', 'pokemon', 'shit', ': 3', 'o.o']...
Trait 'O': ['1', '2', 'u', 'ur', 'cant wait', 'cant', 'wat', 'dont', 'gud', 'wen']...


In [ ]:
# ======================================================
#             HITUNG SKOR UNTUK DATASET
# ======================================================
trait_list = list(lexicon.keys()) if lexicon else ['N', 'A', 'E', 'C', 'O']

# Tampilkan konfigurasi yang sedang digunakan
print(f"\n--- Menghitung skor dengan MAX_LENGTH = {ConfigRB.MAX_LENGTH} ---")

for df_ in [val_df, test_df]:
    # Convert 'Text' to string
    df_['Text'] = df_['Text'].astype(str)

    # Gunakan lambda untuk menerapkan fungsi dengan max_length dari ConfigRB
    scores = df_['Text'].apply(
        lambda text: compute_trait_scores(text, max_length=ConfigRB.MAX_LENGTH)
    )

    score_df = pd.DataFrame(scores.tolist(), columns=trait_list)
    score_df.index = df_.index

    for trait in trait_list:
        df_[f"Score_{trait}"] = score_df[trait]

# Tampilkan contoh skor dan teks pada validation set
print("\nContoh Teks dan Skor yang Dihasilkan (dibatasi oleh MAX_LENGTH):")
columns_to_display = ['Text'] + [f"Score_{t}" for t in trait_list]
print(val_df[columns_to_display].head())


--- Menghitung skor dengan MAX_LENGTH = 128 ---

Contoh Teks dan Skor yang Dihasilkan (dibatasi oleh MAX_LENGTH):
                                                    Text   Score_N   Score_A  \
9627   typemention 5w44w58w9 sxsprapper and dubstep p...  0.000000  0.000000   
11149           i second the motion to gild ucyborgboy95  0.000000  0.000000   
14371  warowl has good quality but he lacks knowledge...  0.000000  0.000000   
2641                       yep it did here is the stream  0.000000  0.000000   
30645  has the damn flu. could the universe's timing ... -0.056042 -0.085421   

       Score_E  Score_C   Score_O  
9627       0.0      0.0  0.000000  
11149      0.0      0.0  0.000000  
14371      0.0      0.0  0.000000  
2641       0.0      0.0  0.000000  
30645      0.0      0.0  0.117249  


# Tuning Threshold (Validation Set)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, matthews_corrcoef

def find_optimal_thresholds_rule_based(
    df,
    trait_list,
    alpha=0.5,
    num_steps=500  # Jumlah langkah pencarian threshold antara min dan max score
):
    optimal_thresholds = {}
    print("Mencari threshold optimal per trait untuk model Rule-Based (mengoptimalkan kombinasi F1 & MCC)...")

    for trait in trait_list:
        score_col = f"Score_{trait}"
        label_col = trait # Asumsi nama kolom label sama dengan nama trait

        # Ekstrak skor dan label untuk trait saat ini
        scores = df[score_col].values
        labels = df[label_col].values

        best_score  = -1
        best_f1     = 0
        best_mcc    = -1
        best_thresh = 0 # Default threshold jika tidak ada yang ditemukan

        # Tentukan rentang pencarian secara dinamis berdasarkan skor di validation set
        min_score = scores.min()
        max_score = scores.max()

        # Jika semua skor sama, pencarian tidak diperlukan.
        if min_score == max_score:
            print(f"  Peringatan untuk trait '{trait}': Semua skor identik ({min_score:.2f}). Threshold tidak dapat dioptimasi.")
            optimal_thresholds[trait] = min_score
            continue

        # Loop melalui kandidat threshold
        for thresh in np.linspace(min_score, max_score, num_steps):
            preds_binary = (scores > thresh).astype(int)

            # Lewati jika prediksi hanya menghasilkan satu kelas (semua 0 atau semua 1)
            if len(np.unique(preds_binary)) < 2:
                continue

            # Hitung F1-score (binary)
            _, _, f1, _ = precision_recall_fscore_support(
                labels, preds_binary, average='binary', zero_division=0
            )
            # Hitung MCC
            mcc = matthews_corrcoef(labels, preds_binary)

            # Normalisasi MCC ke rentang [0, 1] agar sebanding dengan F1
            mcc_norm = (mcc + 1) / 2.0

            # Hitung skor gabungan
            combined_score = alpha * f1 + (1 - alpha) * mcc_norm

            if combined_score > best_score:
                best_score  = combined_score
                best_f1     = f1
                best_mcc    = mcc
                best_thresh = thresh

        optimal_thresholds[trait] = best_thresh
        print(
            f"  {trait.upper()} | "
            f"Threshold={best_thresh:.4f} | "
            f"F1={best_f1:.4f} | "
            f"MCC={best_mcc:.4f} | "
            f"Score_combined={best_score:.4f}"
        )

    return optimal_thresholds

# trait_list = list(lexicon.keys()) # Misal: ['N', 'A', 'E', 'C', 'O']
trait_list = ['O', 'C', 'E', 'A', 'N'] # Sesuaikan urutannya jika perlu

# 2. Jalankan fungsi untuk mencari threshold
#    Memberi bobot yang sama untuk F1 dan MCC (alpha=0.5)
val_thresholds_rb = find_optimal_thresholds_rule_based(
    val_df,
    trait_list=trait_list,
    alpha=0.5,
    num_steps=10000 # Tingkatkan untuk pencarian yang lebih halus
)

# 3. Tampilkan hasil
print("\nOptimal thresholds per trait (Rule-Based):")
for trait, threshold in val_thresholds_rb.items():
    print(f" - {trait}: {threshold:.4f}")

Mencari threshold optimal per trait untuk model Rule-Based (mengoptimalkan kombinasi F1 & MCC)...
  O | Threshold=-0.2230 | F1=0.8405 | MCC=-0.0001 | Score_combined=0.6702
  C | Threshold=-0.0878 | F1=0.4513 | MCC=0.0360 | Score_combined=0.4847
  E | Threshold=-0.0399 | F1=0.5290 | MCC=0.0389 | Score_combined=0.5242
  A | Threshold=-0.0837 | F1=0.5541 | MCC=0.0698 | Score_combined=0.5445
  N | Threshold=-0.1081 | F1=0.5862 | MCC=0.0269 | Score_combined=0.5498

Optimal thresholds per trait (Rule-Based):
 - O: -0.2230
 - C: -0.0878
 - E: -0.0399
 - A: -0.0837
 - N: -0.1081


# Evaluasi Akhir (Test Set)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef
)

def evaluate_rule_based_model(df_test, thresholds_rb, trait_list):
    # 1. DAPATKAN PREDIKSI BINER DAN LABEL ASLI
    # Buat matriks untuk prediksi biner dan label asli
    all_preds_binary = {}
    all_labels = {}

    for trait in trait_list:
        score_col = f"Score_{trait}"
        label_col = trait

        # Terapkan threshold untuk mendapatkan prediksi biner
        scores = df_test[score_col].values
        preds_binary = (scores > thresholds_rb[trait]).astype(int)
        all_preds_binary[trait] = preds_binary

        # Ambil label asli
        all_labels[trait] = df_test[label_col].values

    # Ubah ke format numpy array 2D dengan urutan kolom yang konsisten
    preds_np = pd.DataFrame(all_preds_binary)[trait_list].values
    labels_np = pd.DataFrame(all_labels)[trait_list].values

    # 2. HITUNG METRIK OVERALL (MICRO)
    # - Accuracy micro: flatten semua elemen
    overall_micro_acc = accuracy_score(labels_np.flatten(), preds_np.flatten())
    # - Precision, Recall, F1 micro (gunakan 'binary' karena label sudah 0/1)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels_np.flatten(), preds_np.flatten(), average='binary', zero_division=0
    )
    # - MCC overall (flatten)
    mcc_overall = matthews_corrcoef(labels_np.flatten(), preds_np.flatten())

    # 3. HITUNG METRIK PER DIMENSI
    dim_acc = {}
    precision_per_label = {}
    recall_per_label = {}
    f1_per_label = {}
    mcc_per_label = {}

    for i, trait in enumerate(trait_list):
        labels_dim = labels_np[:, i]
        preds_dim = preds_np[:, i]

        # Handle edge case jika prediksi atau label hanya berisi satu kelas
        if len(np.unique(preds_dim)) < 2 or len(np.unique(labels_dim)) < 2:
            p, r, f = 0.0, 0.0, 0.0
            mcc = 0.0
        else:
            p, r, f, _ = precision_recall_fscore_support(
                labels_dim, preds_dim, average='binary', zero_division=0
            )
            mcc = matthews_corrcoef(labels_dim, preds_dim)

        dim_acc[trait] = accuracy_score(labels_dim, preds_dim)
        precision_per_label[trait] = p
        recall_per_label[trait] = r
        f1_per_label[trait] = f
        mcc_per_label[trait] = mcc

    # 4. TAMPILKAN HASIL OVERALL
    print("---- Test Metrics overall (Rule-Based with Optimized Thresholds) ----")
    print(f"Akurasi   : {overall_micro_acc:.4f}")
    print(f"Presisi   : {precision_micro:.4f}")
    print(f"Recall    : {recall_micro:.4f}")
    print(f"F1-score  : {f1_micro:.4f}")
    print(f"MCC       : {mcc_overall:.4f}\n")

    # 5. TAMPILKAN TABEL METRIK PER DIMENSI
    print("---- Akurasi, Precision, Recall, F1, MCC Tiap Dimensi (Rule-Based) ----")
    print(f"{'Label':<8} | {'Accuracy':>8} | {'Precision':>9} | {'Recall':>7} | {'F1-score':>8} | {'MCC':>7} | {'Threshold':>9}")
    print("-" * 85)
    for trait in trait_list:
        print(
            f"{trait:<8} | "
            f"{dim_acc[trait]:>8.4f} | "
            f"{precision_per_label[trait]:>9.4f} | "
            f"{recall_per_label[trait]:>7.4f} | "
            f"{f1_per_label[trait]:>8.4f} | "
            f"{mcc_per_label[trait]:>7.4f} | "
            f"{thresholds_rb[trait]:>9.4f}"
        )

    # 6. TENTUKAN DIMENSI TERBAIK (VOTING)
    win_counts = {trait: 0 for trait in trait_list}
    if dim_acc:
        best_acc_trait = max(dim_acc, key=dim_acc.get)
        if dim_acc[best_acc_trait] > 0: win_counts[best_acc_trait] += 1
    if f1_per_label:
        best_f1_trait = max(f1_per_label, key=f1_per_label.get)
        if f1_per_label[best_f1_trait] > 0: win_counts[best_f1_trait] += 1
    if mcc_per_label:
        best_mcc_trait = max(mcc_per_label, key=mcc_per_label.get)
        if mcc_per_label[best_mcc_trait] > -1: win_counts[best_mcc_trait] += 1

    if max(win_counts.values()) == 0:
        best_dim_trait = max(f1_per_label, key=f1_per_label.get) if f1_per_label else trait_list[0]
    else:
        best_dim_trait = max(win_counts, key=win_counts.get)

    best_acc_val = dim_acc.get(best_dim_trait, 0)
    best_f1_val  = f1_per_label.get(best_dim_trait, 0)
    best_mcc_val = mcc_per_label.get(best_dim_trait, 0)
    best_thresh_val = thresholds_rb.get(best_dim_trait, 0)

    print(
        f"\nDimensi terbaik diprediksi (berdasarkan voting Accuracy, F1, MCC): "
        f"( {best_dim_trait} ) "
        f"(Akurasi = {best_acc_val:.4f}) "
        f"(F1 = {best_f1_val:.4f}) "
        f"(MCC = {best_mcc_val:.4f}) "
        f"(Threshold = {best_thresh_val:.4f})"
    )

    # 7. RETURN DICTIONARY METRIK
    metrics = {
        'overall_micro_acc': overall_micro_acc,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'f1_micro': f1_micro,
        'mcc_overall': mcc_overall,
        'accuracy_per_dim': dim_acc,
        'precision_per_dim': precision_per_label,
        'recall_per_dim': recall_per_label,
        'f1_per_dim': f1_per_label,
        'mcc_per_dim': mcc_per_label,
        'optimal_thresholds': thresholds_rb,
        'best_dim_by_vote': {
            'name': best_dim_trait,
            'accuracy': best_acc_val,
            'f1': best_f1_val,
            'mcc': best_mcc_val,
            'threshold': best_thresh_val,
            'votes': win_counts.get(best_dim_trait, 0)
        }
    }
    return metrics

trait_list = ['O', 'C', 'E', 'A', 'N']

# Panggil fungsi evaluasi
print("==========================================================")
print("EVALUASI MODEL RULE-BASED PADA TEST SET")
print("==========================================================")

test_metrics_rb = evaluate_rule_based_model(test_df, val_thresholds_rb, trait_list)

EVALUASI MODEL RULE-BASED PADA TEST SET
---- Test Metrics overall (Rule-Based with Optimized Thresholds) ----
Akurasi   : 0.4444
Presisi   : 0.4362
Recall    : 0.9836
F1-score  : 0.6043
MCC       : 0.0575

---- Akurasi, Precision, Recall, F1, MCC Tiap Dimensi (Rule-Based) ----
Label    | Accuracy | Precision |  Recall | F1-score |     MCC | Threshold
-------------------------------------------------------------------------------------
O        |   0.7183 |    0.7200 |  0.9965 |   0.8360 | -0.0201 |   -0.2230
C        |   0.3140 |    0.2987 |  0.9828 |   0.4581 |  0.0460 |   -0.0878
E        |   0.3678 |    0.3524 |  0.9691 |   0.5168 |  0.0359 |   -0.0399
A        |   0.3897 |    0.3721 |  0.9732 |   0.5383 |  0.0626 |   -0.0837
N        |   0.4321 |    0.4282 |  0.9830 |   0.5965 |  0.0160 |   -0.1081

Dimensi terbaik diprediksi (berdasarkan voting Accuracy, F1, MCC): ( O ) (Akurasi = 0.7183) (F1 = 0.8360) (MCC = -0.0201) (Threshold = -0.2230)


# Simpan Metadata

In [ ]:
import json
import os
import numpy as np
from google.colab import files

# 1. Kumpulkan metadata yang relevan untuk model Rule-Based (versi sederhana)
metadata_rb = {
    'model_info': {
        'model_type': 'Rule-Based (Lexicon Search)',
        'description': 'Evaluation using optimal thresholds found on the validation set.',
        'lexicon_path': ConfigRB.LEXICON_PATH,
        'max_length': ConfigRB.MAX_LENGTH
    },
    'dataset_info': {
        'total_data': len(df),
        'train_size': len(train_df),
        'val_size':   len(val_df),
        'test_size':  len(test_df)
    },
    # Bagian ini memiliki struktur yang sama persis dengan output model DL
    'val_thresholds': val_thresholds_rb,
    'test_metrics': test_metrics_rb,
}


# 2. Tentukan nama file yang deskriptif
# Ganti 'NAMA_DATASET' dengan nama yang sesuai
nama_dataset = 'Gabungan_Seluruh_Dataset'
jumlah_data = metadata_rb['dataset_info']['total_data']

# Buat nama file yang informatif dan lebih simpel
metadata_filename_rb = f"v1_rule_based_metadata_{nama_dataset}_JMLH{jumlah_data}.json"
metadata_path_rb = os.path.join('/content/', metadata_filename_rb)


# 3. Simpan metadata ke dalam file JSON
with open(metadata_path_rb, 'w') as f:
    # Menggunakan NpEncoder untuk menangani tipe data numpy
    class NpEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, np.integer):
                return int(obj)
            if isinstance(obj, np.floating):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return super(NpEncoder, self).default(obj)

    json.dump(metadata_rb, f, indent=4, cls=NpEncoder)

print(f"Metadata untuk model Rule-Based berhasil disimpan di: {metadata_path_rb}")


# 4. Download file metadata (opsional, khusus untuk Google Colab)
try:
    files.download(metadata_path_rb)
except Exception as e:
    print(f"Gagal mengunduh file secara otomatis. Silakan unduh manual dari panel file di: {metadata_path_rb}")
    print(f"Error: {e}")

Metadata untuk model Rule-Based berhasil disimpan di: /content/v1_rule_based_metadata_Gabungan_Seluruh_Dataset_JMLH31584.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>